In [0]:
import json, random, uuid
from datetime import datetime, timedelta

# --- Configuration: every knob in one place ---
LANDING = "/Volumes/fleetpulse/raw/landing"   # where "devices" upload their log files
N_DEVICES = 500
DUP_RATE, LATE_RATE, BAD_RATE = 0.03, 0.02, 0.01  # fault injection rates: 3% dupes, 2% late, 1% malformed

random.seed(42)   # same "random" data every run -> reproducible, debuggable
MODELS = ["MRI-3T-Apex", "MRI-1.5T-Core", "CT-128-Slice", "CT-64-Slice"]
HOSPITALS = [("Fortis BLR", "Bangalore"), ("Apollo HYD", "Hyderabad"),
             ("AIIMS DEL", "Delhi"), ("Lilavati MUM", "Mumbai"),
             ("CMC Vellore", "Vellore"), ("Manipal BLR", "Bangalore")]

# --- Build the registry ---
devices = []
for i in range(N_DEVICES):
    h = random.choice(HOSPITALS)
    devices.append({
        "device_id": f"DEV-{i:04d}",          # DEV-0000 ... DEV-0499, zero-padded so sorting works
        "model": random.choice(MODELS),
        "hospital": h[0], "city": h[1],
        "install_date": str((datetime(2020,1,1) + timedelta(days=random.randint(0,2000))).date()),
        "degrading": i % 50 == 0              # every 50th device (10 total) will slowly fail
    })

# --- Persist as a governed Delta table (minus the secret flag) ---
spark.createDataFrame(devices).drop("degrading") \
     .write.mode("overwrite").saveAsTable("fleetpulse.silver.devices")

In [0]:
def make_event(dev, ts, batch_num):
    """One telemetry reading. Degrading devices' vibration climbs with each batch."""
    base_vib = 2.0 + (batch_num * 0.15 if dev["degrading"] else 0)   # slow upward trend = failure signature
    return {
        "event_id": str(uuid.uuid4()),
        "device_id": dev["device_id"],
        "event_ts": ts.isoformat(),
        "coil_temp_c": round(random.gauss(4.2, 0.3), 2),      # MRI coils sit near 4K (liquid helium)
        "helium_pct": round(random.uniform(30, 95), 1),
        "vibration_mm_s": round(random.gauss(base_vib, 0.4), 2),
        "error_code": random.choice([None]*95 + ["E101","E204","E307","E512"] + [None]),  # ~4% carry an error
        "status": "ACTIVE",
        "firmware_version": "fw-2.4.1",
    }

def corrupt(e):
    """Simulate transmission damage: one of three realistic corruption types."""
    c = dict(e)
    kind = random.choice(["null_id", "bad_temp", "bad_ts"])
    if kind == "null_id": c["device_id"] = None          # lost identity -> unjoinable
    elif kind == "bad_temp": c["coil_temp_c"] = -273.0   # physically impossible sensor glitch
    else: c["event_ts"] = "not-a-timestamp"              # unparseable garbage
    return c

def write_batch(batch_num, events_per_batch=20000):
    now = datetime.utcnow()
    events = []
    for _ in range(events_per_batch):
        dev = random.choice(devices)
        ts = now - timedelta(seconds=random.randint(0, 1800))      # events from the last 30 min
        if random.random() < LATE_RATE:
            ts -= timedelta(hours=random.uniform(1, 6))            # a "late" event: happened hours ago, arriving now
        e = make_event(dev, ts, batch_num)
        if random.random() < BAD_RATE: e = corrupt(e)
        events.append(e)
        if random.random() < DUP_RATE: events.append(dict(e))      # duplicate: device re-sent after network timeout
    # one newline-delimited JSON file per batch = one "upload" from the field
    path = f"{LANDING}/batch_{batch_num:04d}_{uuid.uuid4().hex[:6]}.json"
    dbutils.fs.put(path, "\n".join(json.dumps(e) for e in events), overwrite=True)
    print(f"batch {batch_num}: {len(events)} events -> {path}")

for b in range(1, 11):
    write_batch(b)

In [0]:
display(dbutils.fs.ls("/Volumes/fleetpulse/raw/landing"))

In [0]:
print(dbutils.fs.head(dbutils.fs.ls("/Volumes/fleetpulse/raw/landing")[0].path, 500))

In [0]:
%sql
SELECT COUNT(*) AS device_count FROM fleetpulse.silver.devices

In [0]:
%sql
SELECT * FROM fleetpulse.silver.devices LIMIT 10